# Batch-eval review — eight detectors
### Five traditional baselines (incl. two TUNED) · three DINOv3 variants

| # | detector | family | input | source |
|---|----------|--------|-------|--------|
| 1 | Coherent Power             | traditional        | spectrogram | deployed (container) |
| 2 | Power Detection            | traditional        | **raw IQ** (own FFT) | deployed (container) |
| 3 | Power Detection (Tuned)    | traditional, tuned | **raw IQ** (own FFT) | deployed (container, NEW) |
| 4 | Computer Vision            | traditional        | spectrogram | deployed (container) |
| 5 | Computer Vision (Tuned)    | traditional, tuned | spectrogram | deployed (container, NEW) |
| 6 | Zero-shot DINOv3           | learned            | spectrogram | deployed (container) |
| 7 | Fine-tuned DINOv3 (M1: ≤30 dB) | learned        | spectrogram | offline (materialized) |
| 8 | Fine-tuned DINOv3 (M2: all dB) | learned        | spectrogram | offline (materialized) |

This **augments** `../batch_eval_review_six_detectors.ipynb`: it uses the **same metrics,
buckets, thresholds, graph functions, titles, frames, and examples**, and simply adds the
two **tuned** classical baselines — **Power Detection (Tuned)** and **Computer Vision
(Tuned)** — next to their naive counterparts on every plot. The six original detectors'
masks are reused byte-for-byte, so all previously-reported numbers are unchanged.

**What the tuned detectors are** (still fully classical DSP / CV, no ML):
- *Power Detection (Tuned)*: windowed FFT + proper Pfa-driven CFAR on linear power
  (CA/GO/SO) + adaptive per-bin temporal noise floor + DC notch — a *fair* energy
  detector instead of a fixed 6-sigma-in-dB rule.
- *Computer Vision (Tuned)*: per-frequency-column local adaptive threshold + hysteresis
  dual threshold + direction-aware morphological opening + DC notch — a *fair* classical
  CV pipeline instead of a global threshold with a square structuring element.

**Metric consistency (important):** all eight detectors are scored by the *same*
canonical tool, `eval_detector_masks.py`, over the `sweep_20260630` batch run:
- the six deployed/container detectors' masks are their own container outputs (unmodified);
- `finetuned_dino` / `finetuned_dino_m2` masks are the **byte-identical** materialized dirs
  the six-detector notebook uses.

So bucketing (bandwidth/pulse-length from the source SigMF `wfgt:` attributes — e.g.
ZC/METADATA report bandwidth `unknown`), coverage, box-IoU, and the pixel metrics are
computed identically for every detector. `DET_THRESHOLD` matches the original (0.1). Tables
come from `eight_detectors/compare_tables_eight/`, built by `src/assemble_eight_detectors.py`
(see the setup cell + `EIGHT_DETECTOR_WORKFLOW.md`).

> If the two TUNED detectors have not been generated by the container yet, this notebook
> still runs — it shows whichever detectors are present and simply omits the missing ones.
> Re-run `src/assemble_eight_detectors.py` after the container batch run to fill in the
> tuned detectors everywhere.

In [ ]:
from pathlib import Path
import sys, json, subprocess, warnings
import matplotlib.pyplot as plt
from IPython.display import display
warnings.filterwarnings("ignore")

FT_ROOT      = Path.home() / "Holohub-Signal-Detection/dino_fine_tuning"
DINO_REPO    = Path.home() / "dinov3"
EVAL_DIR     = Path.home() / ("Holohub-Signal-Detection/applications/usrp_wideband_signal_detection"
                              "/infocom_evals/signal_detection_experiments")
BATCH_ROOT   = Path("/tmp/usrp_spectrograms/batch_eval/sweep_20260630")   # deployed masks (for panels)
DETS_ROOT    = FT_ROOT / "notebooks/eight_detectors/sweep_detectors_eight"  # combined 8-detector run root
TABLES_DIR   = FT_ROOT / "notebooks/eight_detectors/compare_tables_eight"   # eval_detector_masks output
CAPTURE_DIRS = [Path.home() / "captures"]
DET_THRESHOLD = 0.1   # coverage>=this counts as 'detected' (matches the original; GT are filled boxes)

for p in (DINO_REPO, FT_ROOT / "src", EVAL_DIR):
    sys.path.insert(0, str(p))
import eval_viz as v
import mask_eval_metrics as mem
import plot_eval_results as pe
import finetuned_infer as fi
import yaml

# 8 detectors: five traditional baselines (coherent + naive/tuned power + naive/tuned CV)
# + zero-shot DINO + two fine-tuned variants (M1 <=30 dB, M2 all dB). Order groups the
# traditional family first, each TUNED detector immediately after its naive counterpart
# for direct comparison, then the DINO family.
DET_ORDER = ["coherent_power",
             "power_detection", "power_detection_tuned",
             "computer_vision", "computer_vision_tuned",
             "cuda_dino", "finetuned_dino", "finetuned_dino_m2"]
DET_LABEL = {"coherent_power": "Coherent Power",
             "power_detection": "Power Detection",
             "power_detection_tuned": "Power Detection (Tuned)",
             "computer_vision": "Computer Vision",
             "computer_vision_tuned": "Computer Vision (Tuned)",
             "cuda_dino": "Zero-shot DINOv3",
             "finetuned_dino": "Fine-tuned DINOv3 (M1: ≤30 dB)",
             "finetuned_dino_m2": "Fine-tuned DINOv3 (M2: all dB)"}


def show(fig):
    """Display a figure exactly once (closing it prevents the inline backend from
    also auto-rendering it at cell end -> no duplicate graphs)."""
    display(fig); plt.close(fig)

In [ ]:
# Both fine-tuned detectors (for the live spectrogram panels) + the canonical tables.
train_cfg = yaml.safe_load(open(FT_ROOT / "configs/train.yaml"))
ds_meta   = json.loads((FT_ROOT / "data/dataset/dataset_meta.json").read_text())
detector    = fi.FinetunedDetector(str(FT_ROOT / "checkpoints/M1_ft/best.pt"), train_cfg, ds_meta,
                                   threshold=fi.load_threshold(FT_ROOT / "eval_out/M1_ft/eval_meta.json"))
detector_m2 = fi.FinetunedDetector(str(FT_ROOT / "checkpoints/M2_ft/best.pt"), train_cfg, ds_meta,
                                   threshold=fi.load_threshold(FT_ROOT / "eval_out/M2_ft/eval_meta.json"))
FT_MODELS = {"finetuned_dino": detector, "finetuned_dino_m2": detector_m2}

if not (TABLES_DIR / "region_metrics.csv").exists():
    print("Canonical 8-detector tables missing. Build them with:")
    print(f"  python {FT_ROOT/'src/assemble_eight_detectors.py'}")
    print("(symlinks the 6 container/materialized detectors + the 2 tuned container dirs, then")
    print(f" runs eval_detector_masks.py --coverage-threshold {DET_THRESHOLD} -> compare_tables_eight/)")
    print("The two TUNED deployed detectors (power_detection_tuned, computer_vision_tuned) come")
    print("from the container batch run first -- see eight_detectors/EIGHT_DETECTOR_WORKFLOW.md.")
region = pe.load_region(TABLES_DIR / "region_metrics.csv")
frame  = pe.load_frame(TABLES_DIR / "frame_pixel_metrics.csv")
present = pe._detectors(region)
print(f"detectors present in tables: {present}")
missing = [d for d in DET_ORDER if d not in present]
if missing:
    print(f"NOT YET present (run the container batch + re-assemble to add): {missing}")
print(f"{len(region)} region rows, {len(frame)} frame rows")

In [ ]:
def load_bundle_dets(stem, frame_number):
    """Deployed masks from the sweep (coherent_power, cuda_dino, the naive + tuned
    power/CV detectors -- once generated) + both fine-tuned models run live, pretty-labeled
    and ordered. Any detector whose masks are not present yet is simply omitted."""
    b = v.load_frame_bundle_smart(BATCH_ROOT, frame_number, file_stem=stem, capture_dirs=CAPTURE_DIRS)
    row = next(r for r in mem.load_manifest(BATCH_ROOT / "coherent_power" / stem)
               if int(r["frame_number"]) == frame_number)
    dp = v.find_capture_data(stem, CAPTURE_DIRS)
    iq = v.read_frame_iq(dp, int(row["local_file_offset_complex"]), int(row["complex_samples_read"]))
    for name, mdl in FT_MODELS.items():
        b.detector_masks[name] = fi.to_display_grid(mdl.mask_for_iq(iq), b.fft_rows, b.fft_cols)
    b.detector_masks = {DET_LABEL[k]: b.detector_masks[k] for k in DET_ORDER if k in b.detector_masks}
    return b

## 1. Single-frame overlay (spectrogram + each detector)

In [ ]:
FILE_STEM = "attenuation_dB_45"   # same capture as the original batch_eval_review.ipynb
FRAME = 100                        # same frame number as the original
b = load_bundle_dets(FILE_STEM, FRAME)
print(f"{FILE_STEM} frame {FRAME}: grid {b.fft_rows}x{b.fft_cols}, {len(b.gt_items)} GT regions")
show(v.plot_frame_panels(b, detectors=list(b.detector_masks.keys())))

## 2. Reproducible frame review (annotated + noise-only), all detectors

In [ ]:
REVIEW_STEM = "attenuation_dB_45"   # same capture, same sampling params/seed as the original
SAMPLE = v.sample_review_frames(BATCH_ROOT, REVIEW_STEM, n_annotated=5, n_noise=1, seed=7)
print(f"{SAMPLE['annotated_available']} annotated / {SAMPLE['noise_available']} noise-only; reviewing {SAMPLE['review_frames']}")
for fr in SAMPLE["review_frames"]:
    b = load_bundle_dets(REVIEW_STEM, fr)
    tag = "NOISE-ONLY" if fr in SAMPLE["noise_frames"] else f"{len(b.gt_items)} GT regions"
    print(f"--- frame {fr}: {tag} ---")
    show(v.plot_frame_panels(b, detectors=list(b.detector_masks.keys())))

## 3. Performance vs power, faceted by signal class / bandwidth / pulse length
(Same `plot_eval_results` graphs and titles as the original notebook, one line per detector.)

In [ ]:
show(pe.fig_rate_vs_power_by(region, "signal_class", None, DET_THRESHOLD,
                             "Detection rate vs power, by signal class"))
show(pe.fig_rate_vs_power_by(region, "bandwidth", pe.BW_ORDER, DET_THRESHOLD,
                             "Detection rate vs power, by bandwidth"))
show(pe.fig_rate_vs_power_by(region, "pulse_length", pe.LEN_ORDER, DET_THRESHOLD,
                             "Detection rate vs power, by pulse length (time duration)"))

## 4. Performance (detection rate / box-IoU / coverage) vs bandwidth and vs pulse length

In [ ]:
show(pe.fig_metric_vs_bucket(region, "bandwidth", pe.BW_ORDER, DET_THRESHOLD,
                             "Performance vs signal bandwidth"))
show(pe.fig_metric_vs_bucket(region, "pulse_length", pe.LEN_ORDER, DET_THRESHOLD,
                             "Performance vs pulse length (time duration)"))

## 5. Frame-level accuracy (precision / recall / F1 / pixel-IoU) + false-positive area vs power

In [ ]:
show(pe.fig_frame_metrics_vs_power(frame))

## 6. Numeric summary — detection rate vs attenuation (coverage ≥ threshold)

In [ ]:
import pandas as pd
rdf = pd.DataFrame(region); rdf["detected"] = rdf["coverage"].astype(float) >= DET_THRESHOLD
piv = rdf.pivot_table(index="detector", columns="attenuation_db", values="detected", aggfunc="mean").round(2)
piv = piv.reindex([d for d in DET_ORDER if d in piv.index])
piv.index = [DET_LABEL.get(i, i) for i in piv.index]
print(f"Region detection rate vs attenuation (coverage ≥ {DET_THRESHOLD}):"); display(piv)
cls = rdf.pivot_table(index="signal_class", columns="detector", values="detected", aggfunc="mean").round(2)
cls = cls.reindex(columns=[d for d in DET_ORDER if d in cls.columns])
cls.columns = [DET_LABEL.get(c, c) for c in cls.columns]
print("\nDetection rate by waveform class (all attenuations):"); display(cls)

## 7. Tuned vs naive — did the fair classical baseline close the gap?
Direct paired view: each tuned detector minus its naive counterpart (region detection
rate, coverage ≥ threshold), by attenuation. Positive = the tuned version detects more.

In [ ]:
pairs = [("power_detection", "power_detection_tuned"),
         ("computer_vision", "computer_vision_tuned")]
have = set(pe._detectors(region))
for naive, tuned in pairs:
    if naive in have and tuned in have:
        sub = rdf[rdf["detector"].isin([naive, tuned])]
        p = sub.pivot_table(index="detector", columns="attenuation_db", values="detected", aggfunc="mean")
        delta = (p.loc[tuned] - p.loc[naive]).round(2)
        p = p.reindex([naive, tuned]); p.index = [DET_LABEL[naive], DET_LABEL[tuned]]
        print(f"\n{DET_LABEL[tuned]} vs {DET_LABEL[naive]} — detection rate by attenuation:")
        display(p.round(2))
        print("Δ (tuned − naive):"); display(delta.to_frame(name="Δ detection rate").T)
    else:
        print(f"(skipping {naive} vs {tuned}: one or both not present in tables yet)")

## Caveats (read before quoting numbers)

1. **Metrics are identical across detectors** — all scored by `eval_detector_masks.py`
   over `sweep_20260630`; only the masks differ. Deployed detectors are the container's own
   outputs, unmodified; the six original detectors are byte-identical to the six-detector notebook.
2. **GT includes buried signals** — annotation boxes mark where a transmitter was, even below
   the noise floor at high attenuation; pixel recall/IoU is therefore low for *all* detectors there.
3. **Pixel-IoU favors the fine-tuned model** (trained to reproduce the filled-box convention);
   the sparser energy masks of the traditional detectors are penalized on IoU even when they localize
   the signal. **Region detection rate is the fair headline.**
4. **Geometry** — batch frames are 512×10240 (nfft=10240); the fine-tuned model runs at nfft=1024
   and its mask is MAX-pooled onto the display grid. The classical detectors emit at the batch grid
   and are scored/resized identically to coherent_power.
5. **Raw-IQ vs spectrogram** — both power detectors (naive + tuned) consume raw IQ and run their own
   FFT; both CV detectors + DINO consume the spectrogram. No detector is tuned per file (statistical/
   adaptive thresholds only; the "tuned" detectors are tuned to the *task*, not to any eval file).
6. `DET_THRESHOLD` matches the original notebook (0.1). Rebuild tables via
   `src/assemble_eight_detectors.py` (see the setup cell + `EIGHT_DETECTOR_WORKFLOW.md`).